# Lecture 03 — Logistic Regression for Predictive Analytics (Python)
**Term:** Fall 2025  
**Week/Topic:** Lecture 04 — Logistic Regression  
**Instructor:** Dr. Bushaj  

### What we'll cover (detected from this notebook)
- Probability ↔ odds ↔ log-odds (logit); logistic/sigmoid response
- Maximum Likelihood Estimation (MLE) overview
- `sklearn.linear_model.LogisticRegression` (fit, predict, probabilities)
- `statsmodels` GLM (Binomial) for inference (coefficients / ORs)
- Train/validation/test split; stratification where applicable
- ROC curve and AUC
- Regularization: L1/L2/Elastic Net
- Encode categorical variables (one-hot/dummies)
- Gains and Lift analysis for targeting
- Class imbalance strategies (e.g., `class_weight`)
- Interpreting coefficients in log-odds / odds ratios


## Import Required Libraries


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import matplotlib.pylab as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc


In [ ]:
!pip install dmba
!pip install mord

from mord import LogisticIT
from dmba import classificationSummary, gainsChart, liftChart
from dmba.metric import AIC_score, accuracy_score

## Read Data

In [ ]:
my_drive_path = "YOUR_FILE_PATH_HERE"

In [ ]:
bank_df = pd.read_csv(my_drive_path + 'UniversalBank.csv')

In [ ]:
bank_df

In [ ]:
bank_df.drop(columns=['ID', 'ZIP Code'], inplace=True)
bank_df.columns = [c.replace(' ', '_') for c in bank_df.columns]

In [ ]:
bank_df.columns

## Bank Data - Model with a Single Predictor


In [ ]:
# Define predictors and outcome
predictors = ['Income']  # We will first use only 'Income' as the predictor
outcome = 'Personal_Loan'

In [ ]:
y = bank_df[outcome]
X = bank_df[predictors]

In [ ]:
X

In [ ]:
# Partition data into training and validation sets
train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.4, random_state=1)
print(f"Training set size: {train_X.shape}, Validation set size: {valid_X.shape}")


## Build a Model

In [ ]:
logit_reg = LogisticRegression()


# We are creating a model with default strategies. We can define many different paramters in each of the models we define.
# FOR EXAMPLE:
"""
logit_reg = LogisticRegression(
    penalty="l2",    # Regularization type ('l2' = Ridge regularization, 'l1' = Lasso regularization, 'elasticnet' = combination of both, 'none' = no regularization)
    C=1e42,          # Inverse of regularization strength. Smaller values imply stronger regularization (default is 1.0). A very large value like 1e42 essentially disables regularization.
    solver='liblinear',  # Optimization algorithm to use. Options:
                         #   - 'liblinear': Good for small datasets, supports L1 and L2 penalties, binary classification
                         #   - 'lbfgs': Recommended for multiclass problems (more than 2 classes), supports L2 and none penalties, efficient with large datasets
                         #   - 'newton-cg': Like 'lbfgs', supports L2 and none penalties, good for large datasets
                         #   - 'sag': Stochastic average gradient descent, good for large datasets, supports L2 and none penalties
                         #   - 'saga': Extension of 'sag', supports L1, L2, and elasticnet penalties, also good for large datasets
    max_iter=100,    # Maximum number of iterations the solver will take before stopping. Default is 100. Increase this if your model isn't converging.
    random_state=1,  # Seed for random number generation, used for reproducibility of results.
    class_weight=None,  # Weights associated with classes. Options:
                       #   - None: No class weighting (default)
                       #   - 'balanced': Adjusts weights inversely proportional to class frequencies (useful for imbalanced datasets)
                       #   - dict: Manually specify weights for each class
    multi_class='auto',  # Determines how to handle multiclass classification. Options:
                         #   - 'auto': Chooses 'ovr' (one-vs-rest) for binary classification or small datasets, 'multinomial' for larger datasets
                         #   - 'ovr': One-vs-rest, fits one classifier per class
                         #   - 'multinomial': Fits a single classifier for all classes, works best with 'lbfgs', 'newton-cg', or 'saga' solvers
    verbose=0,        # Controls verbosity of the solver. Set to >0 for more detailed logging of the optimization process.
    tol=1e-4,         # Tolerance for stopping criteria. If the change in the loss function is smaller than `tol`, the algorithm stops.
    fit_intercept=True,  # Whether to fit an intercept term (bias). Default is True. Set to False if your data is already centered.
    intercept_scaling=1, # Only used when `solver='liblinear'`. It scales the intercept when fit_intercept is True.
    n_jobs=None       # Number of CPU cores used for parallel computation. Only relevant for solvers that support parallelism ('sag', 'saga', 'lbfgs').
)

"""




logit_reg.fit(train_X, train_y)

In [ ]:
print(f'Intercept: {logit_reg.intercept_[0]}')
print(f'Coefficient: {logit_reg.coef_[0][0]}')

## Evaluate Model Performance

In [ ]:
# Evaluate model performance
print("Training Set Performance:")
classificationSummary(train_y, logit_reg.predict(train_X))
print("-------------------------------------------------")
print("Validation Set Performance:")
classificationSummary(valid_y, logit_reg.predict(valid_X))

# Calculate and print AIC for the model
print(f'AIC: {AIC_score(valid_y, logit_reg.predict(valid_X), df=len(predictors) + 1)}')

In [ ]:
# Predicting class labels and probabilities for the validation set
# 'predict' returns the predicted class labels (0 or 1)
# 'predict_proba' returns the predicted probabilities for each class (0 and 1)

logit_reg_pred = logit_reg.predict(valid_X)           # Predicted class labels (0 or 1)
logit_reg_proba = logit_reg.predict_proba(valid_X)    # Predicted probabilities for class 0 and class 1


In [ ]:
# Create a DataFrame to store the actual values, predicted probabilities, and predicted class labels
logit_result = pd.DataFrame({
    'actual': valid_y,                               # Actual target variable (0 or 1)
    'p(0)': logit_reg_proba[:, 0],                   # Probability of class 0
    'p(1)': logit_reg_proba[:, 1],                   # Probability of class 1
    'predicted': logit_reg_pred                      # Predicted class label
})

In [ ]:
# display four different cases
interestingCases = [2764, 932, 2721, 702]
print(logit_result.loc[interestingCases])

In [ ]:
# Confusion matrices provide insight into how well the classifier is performing
# They show TP (True Positive), TN (True Negative), FP (False Positive), and FN (False Negative) counts.
# This helps evaluate the model's performance at the default cutoff of 0.5

print("Confusion Matrix for Training Set")
classificationSummary(train_y, logit_reg.predict(train_X))   # Classification summary for training data

print("\nConfusion Matrix for Validation Set")
classificationSummary(valid_y, logit_reg.predict(valid_X))   # Classification summary for validation data

In [ ]:

# Calculate the false positive rate (FPR), true positive rate (TPR), and thresholds
fpr, tpr, thresholds = roc_curve(valid_y, logit_reg_proba[:, 1])

# Calculate the area under the ROC curve (AUC)
roc_auc = auc(fpr, tpr)

# Sort the logistic regression results DataFrame by the predicted probability of class 1 (p(1))
df = logit_result.sort_values(by=['p(1)'], ascending=False)

# Create subplots for the Gains Chart, Lift Chart, and ROC Curve
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 4))

# Plot a Gains Chart on the first subplot (left-hand side)
gainsChart(df.actual, ax=axes[0])
axes[0].set_title("Gains Chart")  # Set title for Gains Chart

# Plot a Lift Chart on the second subplot (middle)
liftChart(df['p(1)'], title=False, ax=axes[1])
axes[1].set_title("Lift Chart")  # Set title for Lift Chart

# Plot ROC Curve on the third subplot (right-hand side)
axes[2].plot(fpr, tpr, color='blue', label=f'ROC curve (area = {roc_auc:.2f})')
axes[2].plot([0, 1], [0, 1], color='red', linestyle='--')  # Diagonal line for random chance
axes[2].set_xlabel('False Positive Rate')
axes[2].set_ylabel('True Positive Rate')
axes[2].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[2].legend(loc='lower right')

# Adjust the layout of the plots to prevent overlap and improve the appearance of the figure
plt.tight_layout()

# Display the plots
plt.show()


## Another way to find a good cut-off value

In [ ]:
# Create a DataFrame to hold the predicted probabilities from the logistic regression model
pred_proba_df = pd.DataFrame(logit_reg.predict_proba(valid_X))

# Define a list of threshold values to evaluate model predictions
threshold_list = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45,
                  0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]

# Iterate through each threshold to assess model performance
for i in threshold_list:
    print('\n******** For threshold = {:.2f} ******'.format(i))  # Display the current threshold being evaluated

    # Apply the threshold to the predicted probabilities to generate binary predictions
    # If probability > threshold, assign 1 (positive class), else assign 0 (negative class)
    # We use .iloc[:, 1] to focus on the probabilities for the positive class
    Y_test_pred = (pred_proba_df.iloc[:, 1] > i).astype(int)  # Create binary predictions directly

    # Calculate the accuracy of the predictions compared to the actual values
    test_accuracy = accuracy_score(valid_y, Y_test_pred)

    # Print the calculated accuracy for the current threshold
    print('Our testing accuracy is {:.4f}'.format(test_accuracy))

    # Display a detailed classification summary for the predictions
    print(classificationSummary(valid_y, Y_test_pred))


## Building the Full Model

In [ ]:
#read the data again (to avoid changes we did above)
bank_df = pd.read_csv(my_drive_path + 'UniversalBank.csv')
bank_df.drop(columns=['ID', 'ZIP Code'], inplace=True)
bank_df.columns = [c.replace(' ', '_') for c in bank_df.columns]

In [ ]:
bank_df['Education'] = bank_df['Education'].astype('category')
new_categories = {1: 'Undergrad', 2: 'Graduate', 3: 'Advanced/Professional'}
bank_df['Education'] = bank_df['Education'].cat.rename_categories(new_categories)

In [ ]:
bank_df

In [ ]:
# One-hot only 'Education' (drop_first avoids dummy trap)
bank_df = pd.get_dummies(bank_df, columns=['Education'], prefix_sep='_', drop_first=True)

In [ ]:
bank_df

In [ ]:
y = bank_df['Personal_Loan'].astype(int)
X = bank_df.drop(columns=['Personal_Loan'])

In [ ]:
train_X, valid_X, train_y, valid_y = train_test_split(
    X, y, test_size=0.40, random_state=1, stratify=y
)

In [ ]:
logit_reg = LogisticRegression(
    penalty="l2", C=10.0, solver='lbfgs', max_iter=1000, class_weight=None
)
logit_reg.fit(train_X, train_y)

In [ ]:
print('Intercept:', float(logit_reg.intercept_))
print(pd.DataFrame({'Coefficient': logit_reg.coef_[0]}, index=X.columns))

print("-------------------------------------------------")

In [ ]:
classificationSummary(train_y, logit_reg.predict(train_X))
print("-------------------------------------------------")
classificationSummary(valid_y, logit_reg.predict(valid_X))

### Proper AIC/BIC + p-values (statsmodels)

In [ ]:


# --- 0) Ensure target is numeric 0/1 ---
train_y = train_y.astype(int)
valid_y = valid_y.astype(int)

# --- 1) Keep only numeric columns and coerce everything to float ---
def to_numeric_frame(df):
    # Try numeric coercion for all columns; non-numeric becomes NaN
    out = df.apply(pd.to_numeric, errors='coerce')
    # Optional: drop columns that are entirely NaN after coercion
    all_nan_cols = out.columns[out.isna().all()]
    if len(all_nan_cols):
        print("Dropping all-NaN columns after coercion:", list(all_nan_cols))
        out = out.drop(columns=all_nan_cols)
    return out

train_X_num = to_numeric_frame(train_X)
valid_X_num = to_numeric_frame(valid_X)

# --- 2) Handle missing values (simple median impute; pick your strategy) ---
median_vals = train_X_num.median()
train_X_num = train_X_num.fillna(median_vals)
valid_X_num = valid_X_num.fillna(median_vals)

# --- 3) Optional: drop zero-variance columns (helps stability) ---
nunique = train_X_num.nunique()
zero_var_cols = nunique.index[nunique <= 1]
if len(zero_var_cols):
    print("Dropping zero-variance columns:", list(zero_var_cols))
    train_X_num = train_X_num.drop(columns=zero_var_cols)
    valid_X_num = valid_X_num.drop(columns=zero_var_cols, errors='ignore')

# --- 4) Build SM design matrices as float64 + add constant ---
X_train_sm = sm.add_constant(train_X_num.astype(float), has_constant='add')
X_valid_sm = sm.add_constant(valid_X_num.astype(float), has_constant='add')

# --- 5) Final sanity checks (these should print empty) ---
print("Object cols in X_train_sm:", list(X_train_sm.select_dtypes(include='object').columns))
print("Any NaNs in X_train_sm?", X_train_sm.isna().any().any())
print("Any NaNs in train_y?", pd.isna(train_y).any())

# --- 6) Fit Logit ---
sm_model = sm.Logit(train_y.astype(float), X_train_sm).fit(disp=False)
print(sm_model.summary2())
print({'AIC': sm_model.aic, 'BIC': sm_model.bic})


The intercept is −13.6. Remember: in log-odds space, that’s the baseline log-odds of taking a loan when all features = 0 (not directly meaningful for centered data, but it anchors the curve).

Each coefficient is a change in log-odds per unit. If I exponentiate a coefficient, I get an odds ratio.


Significant positive predictors (p < 0.05):

Income (0.0675, p<0.001) → OR ≈ exp(0.0675)=1.07 per unit.
“Every +1 unit in Income increases odds by ~7% (check your units—if income is in $1k, then +$10k ≈ OR exp(0.675)≈1.96).”

Family (0.623, p<0.001) → OR ≈ 1.86.
“Larger family size is associated with higher odds of accepting the loan.”

CD_Account (3.59, p<0.001) → OR ≈ 36.2.
“CD holders are much more likely to accept—the model thinks they’re prime targets.”

Education_Graduate (4.41, p<0.001) and Education_Advanced/Professional (4.61, p<0.001) → ORs ≈ 82 and 100.
“Higher education levels are strongly associated with acceptance in this dataset.”

Securities_Account (−0.99, p=0.013), Online (−0.73, p=0.001), CreditCard (−1.10, p<0.001) → ORs < 1.
“Customers using online banking or carrying the bank’s credit card show lower odds here—could be segment behavior (already satisfied with current products).”

Borderline / non-sig:

CCAvg (p≈0.057) borderline. “Likely positive but we’d be cautious.”

Age, Experience, Mortgage not significant here.
“Age and Experience are often correlated—collinearity can blur p-values. For prediction that’s okay; for inference we’d check VIF or regularize.”

Model fit lines:

“Pseudo R² ≈ 0.65 indicates strong separation on this dataset.”

“AIC/BIC are for model comparison; lower is better. Keep them for when we try feature changes.

### Cutoff selection demos (J / F1 / cost-optimal)

There are different (perfectly defensible) ways to pick a probability cutoff for turning scores into 0/1
Some commonly used are:
Youden’s J (a “balanced errors” cutoff)
  - pick the threshold that maximizesJ = TPR − FPR (sensitivity − false-positive-rate).
  How to get it: compute ROC, find the thr where (TPR−FPR) is largest.
  When to use it: costs of FP and FN are roughly similar and you want a single, neutral operating point.
  
**YoudenJ is the fairest referee: it picks the cutoff that maximizes ‘true-hits minus false alarms.’**


F1-optimal (maximize the F1 score at some cutoff)

  - pick the threshold that maximizes F1 = 2 · (Precision · Recall) / (Precision + Recall).
  How to get it: sweep thresholds; at each, compute precision & recall; choose the thr with the highest F1.
  When to use it: you want a single number balancing “catching positives” (recall) with “being right when you say yes” (precision), especially on imbalanced data.

**F1-opt picks the cutoff where the teamwork of precision and recall is best.**


Cost-optimal (minimize expected misclassification cost)
  - choose the threshold that minimizes
  Expected cost = FP · C_FP + FN · C_FN (usually TP/TN cost = 0).
  How to get it in practice: either use that closed-form or evaluate cost on a grid of thresholds on your validation set and pick the minimum—this accounts for sampling noise and any extra constraints you add.

  When to use it: you actually know (or can proxy) the business costs/asymmetry.

**Cost-opt is the CFO’s cutoff: we pick the threshold that makes the fewest expensive mistakes.**


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, average_precision_score, confusion_matrix


p = pd.Series(logit_reg.predict_proba(valid_X)[:, 1], index=valid_y.index)

# Youden’s J
fpr, tpr, thr_roc = roc_curve(valid_y, p)
thr_j = thr_roc[(tpr - fpr).argmax()]

# F1-optimal
prec, rec, thr_pr = precision_recall_curve(valid_y, p)
f1s = 2*prec*rec/(prec+rec+1e-12)
thr_f1 = thr_pr[np.argmax(f1s[:-1])]

# Cost-optimal (you can edit these!)
FP_COST, FN_COST = 1.0, 5.0
grid = np.linspace(0.01, 0.99, 99)
def expected_cost(th):
    yhat = (p >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(valid_y, yhat).ravel()
    return fp*FP_COST + fn*FN_COST
thr_cost = grid[np.argmin([expected_cost(t) for t in grid])]

for name, thr in [('0.50', 0.50), ('YoudenJ', thr_j), ('F1-opt', thr_f1), ('Cost-opt', thr_cost)]:
    yhat = (p >= thr).astype(int)
    print(f'\n[{name}] thr={thr:.3f}  Acc={accuracy_score(valid_y, yhat):.3f}  '
          f'Prec={precision_score(valid_y, yhat, zero_division=0):.3f}  '
          f'Rec={recall_score(valid_y, yhat):.3f}  F1={f1_score(valid_y, yhat):.3f}')

0.50 → Acc .960 | Prec .889 | Rec .667 | F1 .762
**Balanced-ish, but misses 1/3 of positives.**

YoudenJ (thr≈0.091) → Acc .894 | Prec .473 | Rec .875 | F1 .614
**Great catch-rate (recall), but many false alarms. Use when missing positives is very costly.**


F1-opt (thr≈0.562) → Acc .963 | Prec .934 | Rec .661 | F1 .774
**Best harmonic balance of precision/recall on this data. Slightly higher threshold than 0.5, gets a tiny precision bump, recall similar.**


Cost-opt (thr≈0.230; FN cost 5× FP) → Acc .943 | Prec .668 | Rec .797 | F1 .727
**When false negatives are 5× worse, threshold drops, recall rises, precision falls. This ties metrics to business cost.**



**There is no sacred 0.50. Pick the operating point by business goal: catch more (YoudenJ/low thr), stay precise (higher thr), or minimize expected cost (cost-opt).**

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# --- Plotting Comparisons ---
# Youden's J (from ROC)
fpr, tpr, thr_roc = roc_curve(valid_y, p)
thr_j = float(thr_roc[(tpr - fpr).argmax()])

# F1-opt (from PR sweep)
prec_curve, rec_curve, thr_pr = precision_recall_curve(valid_y, p)
f1_curve = 2*prec_curve*rec_curve/(prec_curve+rec_curve+1e-12)
thr_f1 = float(thr_pr[np.argmax(f1_curve[:-1])])

# Cost-opt (feel free to play with the costs, the idea is that you can represent the actual costs of your business)
FP_COST, FN_COST = 1.0, 5.0
grid = np.linspace(0.01, 0.99, 99)
def expected_cost(th):
    yhat = (p >= th).astype(int)
    tn, fp, fn, tp = (
        ((yhat==0)&(valid_y==0)).sum(),
        ((yhat==1)&(valid_y==0)).sum(),
        ((yhat==0)&(valid_y==1)).sum(),
        ((yhat==1)&(valid_y==1)).sum(),
    )
    return fp*FP_COST + fn*FN_COST
thr_cost = float(grid[np.argmin([expected_cost(t) for t in grid])])

# Put them in a dict for easy iteration
ops = {
    "YoudenJ": thr_j,
    "F1-opt": thr_f1,
    "Cost-opt": thr_cost,
}

# --- sweep metrics vs threshold for plotting ---
sweep = np.linspace(0.01, 0.99, 199)
precisions, recalls, f1s = [], [], []
for thr in sweep:
    yhat = (p >= thr).astype(int)
    precisions.append(precision_score(valid_y, yhat, zero_division=0))
    recalls.append(recall_score(valid_y, yhat))
    f1s.append(f1_score(valid_y, yhat))
thr_df = pd.DataFrame({
    "threshold": sweep,
    "precision": precisions,
    "recall": recalls,
    "f1": f1s
})

# --- helper to compute metrics at a specific threshold (for hover text) ---
def metrics_at(thr):
    yhat = (p >= thr).astype(int)
    return (
        precision_score(valid_y, yhat, zero_division=0),
        recall_score(valid_y, yhat),
        f1_score(valid_y, yhat)
    )

# --- build the figure ---
fig_thr = go.Figure()
fig_thr.add_trace(go.Scatter(x=thr_df['threshold'], y=thr_df['precision'],
                             name='Precision', mode='lines'))
fig_thr.add_trace(go.Scatter(x=thr_df['threshold'], y=thr_df['recall'],
                             name='Recall', mode='lines'))
fig_thr.add_trace(go.Scatter(x=thr_df['threshold'], y=thr_df['f1'],
                             name='F1', mode='lines'))

# Add vertical reference lines + invisible (thin) scatter for hover tooltips
for label, thr in ops.items():
    prc, rcl, f1v = metrics_at(thr)
    # vertical line (shape)
    fig_thr.add_vline(x=thr, line_width=2, line_dash='dash', line_color='gray')
    # scatter line to enable hover tooltip
    fig_thr.add_trace(
        go.Scatter(
            x=[thr, thr], y=[0, 1],
            mode='lines',
            line=dict(width=12, color='rgba(0,0,0,0)'),  # invisible but thick hitbox
            showlegend=False,
            hovertemplate=(
                f"<b>{label}</b><br>"
                f"Threshold: {thr:.3f}<br>"
                f"Precision: {prc:.3f}<br>"
                f"Recall: {rcl:.3f}<br>"
                f"F1: {f1v:.3f}<extra></extra>"
            )
        )
    )
    # label at top
    fig_thr.add_annotation(
        x=thr, y=1.02, xref='x', yref='paper', showarrow=False,
        text=f"{label}<br>{thr:.3f}", align='center', font=dict(size=10)
    )

fig_thr.update_layout(
    title='Metrics vs Threshold with Operating Points',
    xaxis_title='Threshold',
    yaxis_title='Metric value (0–1)',
    hovermode='x unified'
)
fig_thr.show()


### PR curve + AUPRC (pairs with your ROC)

In [ ]:


ap = average_precision_score(valid_y, p)
pr_prec, pr_rec, _ = precision_recall_curve(valid_y, p)

plt.figure(figsize=(5,4))
plt.step(pr_rec, pr_prec, where='post')
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title(f'PR curve (AP = {ap:.3f})')
plt.show()


This PR curve shows the trade-off between recall (how many real positives we catch) and precision (how often we’re right when we say ‘positive’).

A curve that hugs the top-right means we can get high recall without precision collapsing.”

**“If the curve drops steeply, it means once we try to catch more positives, we quickly start pulling in many negatives (precision falls). That’s a warning for large-scale outreach.**

### Cumulative Gains, Lift, and Precision@K (budgeted actions)

In [ ]:
def gains_lift(y_true, p_hat, n_bins=10):
    order = np.argsort(-p_hat)
    y_sorted = y_true.values[order]
    bins = np.array_split(y_sorted, n_bins)
    pos_each = np.array([b.sum() for b in bins])
    cum_pos = pos_each.cumsum()
    P = y_true.sum()
    df_gl = pd.DataFrame({
        'decile': np.arange(1, n_bins+1),
        'cum_pct_population': np.arange(1, n_bins+1)/n_bins*100,
        'cum_gain_%': (cum_pos / P) * 100.0,
        'lift_each_bin': pos_each / (P/n_bins)
    })
    return df_gl

gl = gains_lift(valid_y, p.values, n_bins=10)
print(gl.head())

def topk_metrics(y_true, p_hat, k_frac=0.10):
    k = max(1, int(len(p_hat)*k_frac))
    idx = np.argsort(-p_hat)[:k]
    topk_y = y_true.iloc[idx]
    prec_at_k = topk_y.mean()
    rec_at_k = topk_y.sum()/y_true.sum()
    baseline = y_true.mean()
    lift_at_k = (prec_at_k / baseline) if baseline > 0 else np.nan
    return prec_at_k, rec_at_k, lift_at_k

p_at_10, r_at_10, lift_10 = topk_metrics(valid_y, p.values, 0.10)
print(f'Precision@10%={p_at_10:.3f}  Recall@10%={r_at_10:.3f}  Lift@10%={lift_10:.2f}x')


If we can only act on the top 10% of customers (budget/capacity), we recover 75% of all positives immediately. That’s what the cumulative gain = 75% at 10% means.   

Within that top decile, 72% are actually positive (Precision@10% = 0.72).

That’s 7.5× better than random (Lift@10% = 7.5×). Random targeting would only hit the base rate.

From these numbers we can back out the base rate (prevalence):
baseline ≈ precision@10 / lift@10 ≈ 0.72 / 7.5 ≈ 9.6% positives overall.”

“On 3,000 customers, that’s ~288 positives overall.
In the top 10% = 300 people, we’d expect ~216 positives (0.72×300).
That’s 75% of the 288 (216/288), consistent with the cum_gain = 75%.”

“Randomly calling 300 people at 9.6% base rate would net ~29 positives.
The model nets ~216 → 7.5× improvement.


If we can only call 10%, the right report is not ‘accuracy’ at a global 0.50 threshold, but Precision@K, Recall@K, and Lift@K at the budget K.”

“If the budget increases (say to 20%), we move to the next decile, note how lift_each_bin drops (1.25× in decile 2). Diminishing returns are visible before we spend.


**PR/AP: “AP tells me how well I stay precise while chasing recall in a rare-positive world.**

**Gains: “At 10% effort, we capture 75% of the prize.”**

**Lift: “Top decile performs 7.5× better than random—fastest path to ROI.”**

**Budgeted metrics: “For fixed capacity, report Precision@K / Recall@K / Lift@K—not global accuracy.”**



In [ ]:
# Build the gains/lift dataframe from your validation labels and probs
# Above I built them separate, but here is a way to build them together as well.

def gains_lift_df(y_true, p_hat, n_bins=10):
    # Ensure numpy arrays
    y_true = np.asarray(y_true).astype(int)
    p_hat = np.asarray(p_hat).astype(float)

    # Sort by descending score
    order = np.argsort(-p_hat)
    y_sorted = y_true[order]

    # Split into deciles (or n_bins)
    bins = np.array_split(y_sorted, n_bins)
    pos_each = np.array([b.sum() for b in bins])

    P = y_true.sum()
    if P == 0:
        raise ValueError("No positives in y_true — lift is undefined.")

    expected_each = P / n_bins
    lift_each = pos_each / expected_each
    cum_pos = pos_each.cumsum()
    cum_gain = (cum_pos / P) * 100.0

    return pd.DataFrame({
        'decile': np.arange(1, n_bins + 1),
        'lift_each_bin': lift_each,
        'cum_gain_%': cum_gain
    })

# Create gl_df from your existing variables (valid_y, p)
# p should be probs for the positive class on the validation set:
# p = logit_reg.predict_proba(valid_X)[:, 1]
gl_df = gains_lift_df(valid_y, p, n_bins=10)

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig_dual = make_subplots(specs=[[{"secondary_y": True}]])
fig_dual.add_trace(
    go.Bar(x=gl_df['decile'], y=gl_df['lift_each_bin'], name='Lift (bin)'),
    secondary_y=False
)
fig_dual.add_trace(
    go.Scatter(x=gl_df['decile'], y=gl_df['cum_gain_%'], name='Cumulative Gain (%)'),
    secondary_y=True
)

fig_dual.update_layout(title='Lift (bars) and Cumulative Gain (line)')
fig_dual.update_xaxes(title_text='Decile')
fig_dual.update_yaxes(title_text='Lift (×)', secondary_y=False)
fig_dual.update_yaxes(title_text='Cumulative Gain (%)', secondary_y=True)
fig_dual.show()


## Grid Search  - Building the Perfect Model


**did not cover in class - will come back to GridSearch soon**

In [ ]:
#default model:
logit_reg = LogisticRegression()
logit_reg.fit(train_X, train_y)


In [ ]:
# -- some imports we don't have above
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score

In [ ]:
# Pipeline: scale -> logistic (scaling helps saga/lbfgs; liblinear is fine too)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000, n_jobs=-1))
])

# Search space: liblinear (L1/L2), lbfgs (L2), saga (L1/L2/ElasticNet with l1_ratio)
param_grid = [
    {
        'model__solver': ['liblinear'],
        'model__penalty': ['l1','l2'],
        'model__C': [0.03, 0.1, 0.3, 1, 3, 10],
        'model__class_weight': [None, 'balanced']
    },
    {
        'model__solver': ['lbfgs'],
        'model__penalty': ['l2'],
        'model__C': [0.03, 0.1, 0.3, 1, 3, 10],
        'model__class_weight': [None, 'balanced']
    },
    {
        'model__solver': ['saga'],
        'model__penalty': ['l1','l2','elasticnet'],
        'model__l1_ratio': [0.1, 0.5, 0.9],
        'model__C': [0.03, 0.1, 0.3, 1, 3, 10],
        'model__class_weight': [None, 'balanced']
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gs = GridSearchCV(
    pipe, param_grid=param_grid,
    scoring='average_precision',   # PR-AUC
    cv=cv, n_jobs=-1, refit=True, verbose=1
)

gs.fit(train_X, train_y)

print("Best CV AP (PR-AUC):", gs.best_score_)
print("Best params:", gs.best_params_)
best_model = gs.best_estimator_   # pipeline with scaler + tuned LR


In [ ]:
def youden_threshold(y_true, p_hat):
    fpr, tpr, thr = roc_curve(y_true, p_hat)
    j = tpr - fpr
    return float(thr[np.argmax(j)])

def f1_opt_threshold(y_true, p_hat):
    prec, rec, thr = precision_recall_curve(y_true, p_hat)
    f1 = 2*prec*rec/(prec+rec+1e-12)
    # last point has no threshold
    return float(thr[np.argmax(f1[:-1])])

def topk_metrics(y_true, p_hat, k_frac=0.10):
    k = max(1, int(len(p_hat)*k_frac))
    idx = np.argsort(-p_hat)[:k]
    top = y_true.iloc[idx]
    prec_k = top.mean()
    rec_k  = top.sum()/y_true.sum()
    base   = y_true.mean()
    lift_k = (prec_k / base) if base>0 else np.nan
    return float(prec_k), float(rec_k), float(lift_k)

def eval_model(name, model, X, y):
    """Return a dict of key metrics & recommended thresholds for a model."""
    p = model.predict_proba(X)[:,1]
    # global metrics (ranking)
    roc = roc_auc_score(y, p)
    ap  = average_precision_score(y, p)

    # fixed 0.50
    y50 = (p >= 0.50).astype(int)
    acc50 = accuracy_score(y, y50)
    f150  = f1_score(y, y50)
    pr50  = precision_score(y, y50, zero_division=0)
    rc50  = recall_score(y, y50)

    # threshold selection
    thr_j  = youden_threshold(y, p)
    thr_f1 = f1_opt_threshold(y, p)

    # compute F1 at those thresholds
    def met_at(thr):
        yh = (p >= thr).astype(int)
        return (precision_score(y, yh, zero_division=0),
                recall_score(y, yh),
                f1_score(y, yh))

    pr_j, rc_j, f1_j   = met_at(thr_j)
    pr_o, rc_o, f1_o   = met_at(thr_f1)

    # budgeted @10%
    p_at10, r_at10, lift10 = topk_metrics(y, p, 0.10)

    return {
        'name': name,
        'roc_auc': roc, 'ap': ap,
        'acc@0.50': acc50, 'f1@0.50': f150, 'prec@0.50': pr50, 'rec@0.50': rc50,
        'thr_YoudenJ': thr_j, 'prec@J': pr_j, 'recall@J': rc_j, 'f1@J': f1_j,
        'thr_F1opt': thr_f1, 'prec@F1opt': pr_o, 'recall@F1opt': rc_o, 'f1@F1opt': f1_o,
        'Precision@10%': p_at10, 'Recall@10%': r_at10, 'Lift@10%': lift10,
        'probs': p  # keep for plotting later
    }


In [ ]:
# Baseline = the defaul model we built above (no scaling in pipeline)
baseline_eval = eval_model("Baseline LR", logit_reg, valid_X, valid_y)

# Tuned = best pipeline from GridSearchCV
tuned_eval = eval_model("Tuned LR (GS)", best_model, valid_X, valid_y)

pd.DataFrame([baseline_eval, tuned_eval]).drop(columns=['probs'])


In [ ]:
def dmba_summaries(model, name, X, y):
    print(f"\n================ {name} ================")
    p = model.predict_proba(X)[:, 1]

    thresholds = {
        "@0.50"   : 0.50,
        "@YoudenJ": youden_threshold(y, p),
        "@F1-opt" : f1_opt_threshold(y, p),
    }

    for label, thr in thresholds.items():
        yhat = (p >= thr).astype(int)
        print(f"\n--- classificationSummary {label} (threshold={thr:.3f}) ---")
        classificationSummary(y, yhat)

# Run for both models on the SAME validation set
dmba_summaries(logit_reg, "Baseline Logistic Regression", valid_X, valid_y)
dmba_summaries(best_model, "Tuned Logistic Regression (GridSearchCV)", valid_X, valid_y)

In [ ]:

# unpack
p_base = baseline_eval['probs']
p_tune = tuned_eval['probs']

# ROC
fpr_b, tpr_b, _ = roc_curve(valid_y, p_base)
fpr_t, tpr_t, _ = roc_curve(valid_y, p_tune)

fig_roc = go.Figure()
fig_roc.add_scatter(x=fpr_b, y=tpr_b, mode='lines',
                    name=f"Baseline (AUC={baseline_eval['roc_auc']:.3f})")
fig_roc.add_scatter(x=fpr_t, y=tpr_t, mode='lines',
                    name=f"Tuned (AUC={tuned_eval['roc_auc']:.3f})")
fig_roc.add_scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash'))
fig_roc.update_layout(title='ROC: Baseline vs Tuned', xaxis_title='FPR', yaxis_title='TPR')
fig_roc.show()

# PR
pr_b, rc_b, _ = precision_recall_curve(valid_y, p_base)
pr_t, rc_t, _ = precision_recall_curve(valid_y, p_tune)

fig_pr = go.Figure()
fig_pr.add_scatter(x=rc_b, y=pr_b, mode='lines',
                   name=f"Baseline (AP={baseline_eval['ap']:.3f})")
fig_pr.add_scatter(x=rc_t, y=pr_t, mode='lines',
                   name=f"Tuned (AP={tuned_eval['ap']:.3f})")
fig_pr.update_layout(title='PR: Baseline vs Tuned', xaxis_title='Recall', yaxis_title='Precision')
fig_pr.show()


In [ ]:
def lift_at_k(y_true, p_hat, k_frac=0.10):
    k = max(1, int(len(p_hat)*k_frac))
    idx = np.argsort(-p_hat)[:k]
    prec_k = y_true.iloc[idx].mean()
    base   = y_true.mean()
    return float(prec_k / base) if base>0 else np.nan

for k in [0.05, 0.10, 0.20]:
    print(f"K={int(k*100)}% | Lift baseline={lift_at_k(valid_y, p_base, k):.2f}× "
          f"vs tuned={lift_at_k(valid_y, p_tune, k):.2f}×")
